# Fixed-report source and claim assessment
Reassesses the frozen version-2 claims without generating replacements. Vendor-domain eligibility and model-assisted review are separate from independent human review. No reference verdicts or API keys are included in the runtime input.


In [ ]:
import subprocess
import sys
from pathlib import Path

DATA = next(p.parent for p in Path('/kaggle/input').rglob('report-v2.json'))
wheel = next(DATA.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel), 'transformers==4.57.6'], check=True)


In [ ]:
import importlib.metadata
import json
import time

import torch

from marketlens.audit import audit_report, report_digest
from marketlens.models import ReportBundle, Settings
from marketlens.providers import QwenModel
from marketlens.report import render, report_summary

assert torch.cuda.is_available()
torch.manual_seed(42)
original = ReportBundle.model_validate_json((DATA / 'report-v2.json').read_text())
started = time.monotonic()
model = QwenModel(Settings().model)
reviewed, decisions = audit_report(original, model)
OUT = Path('/kaggle/working/audit')
OUT.mkdir(exist_ok=True)
(OUT / 'report.json').write_text(reviewed.model_dump_json(indent=2))
(OUT / 'report.md').write_text(render(reviewed))
(OUT / 'assessments.json').write_text(json.dumps(decisions, indent=2))
runtime = {'gpu': torch.cuda.get_device_name(0), 'duration_seconds': time.monotonic() - started, 'parent_report_sha256': report_digest(original), 'summary': report_summary(reviewed), 'model_revision': model.model_revision, 'input_tokens': model.input_tokens, 'output_tokens': model.output_tokens, 'versions': {name: importlib.metadata.version(name) for name in ['torch', 'transformers', 'marketlens']}}
(OUT / 'runtime.json').write_text(json.dumps(runtime, indent=2))
print(json.dumps(runtime, indent=2), flush=True)
